# Washington State Electric Vehicle Population Analysis

## Project Overview
This project provides a data-driven dashboard for a regional electrical utility company to plan infrastructure upgrades. By analyzing EV registration density and vehicle types, the utility can proactively manage grid load and strategically place public charging stations.

### Business Objectives:
1. **Identify High-Growth Clusters:** Locate zip codes with high EV density using geographic mapping.
2. **Infrastructure Planning:** Segment vehicles by battery type (BEV vs. PHEV) and CAFV eligibility to estimate power demand.
3. **Market Sentiment:** Track brand dominance to understand consumer trends in the regional market.

In [1]:
import pandas as pd
import numpy as np

# Load the dataset
file_path = '../datasets/raw_dataset/Electric_Vehicle_Population_Data.csv'
df = pd.read_csv(file_path)

# Quick overview
display(df.head())
print(df.info())

,VIN (1-10),County,City,State,Postal Code,Model Year,Make,Model,Electric Vehicle Type,Clean Alternative Fuel Vehicle (CAFV) Eligibility,Electric Range,Legislative District,DOL Vehicle ID,Vehicle Location,Electric Utility,2020 Census Tract
0,1N4AZ0CP6D,King,Kirkland,WA,98034.0,2013,NISSAN,LEAF,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,75.0,1.0,154635729,POINT (-122.22901 47.72201),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303302e+10
1,5YJ3E1EC8L,Kitsap,Bainbridge Island,WA,98110.0,2020,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,308.0,23.0,107518645,POINT (-122.521 47.62759),PUGET SOUND ENERGY INC,5.303509e+10
2,5YJ3E1EBXJ,King,Seattle,WA,98144.0,2018,TESLA,MODEL 3,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,215.0,37.0,474808813,POINT (-122.30866 47.57874),CITY OF SEATTLE - (WA)|CITY OF TACOMA - (WA),5.303301e+10
3,ZFAFFAC45R,Thurston,Yelm,WA,98597.0,2024,FIAT,500E,Battery Electric Vehicle (BEV),Eligibility unknown as battery range has not b...,0.0,2.0,273658514,POINT (-122.60735 46.94239),PUGET SOUND ENERGY INC,5.306701e+10
4,5YJYGDEE3L,King,Kent,WA,98030.0,2020,TESLA,MODEL Y,Battery Electric Vehicle (BEV),Clean Alternative Fuel Vehicle Eligible,291.0,47.0,109579900,POINT (-122.19975 47.37483),PUGET SOUND ENERGY INC||CITY OF TACOMA - (WA),5.303303e+10


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 280833 entries, 0 to 280832
Data columns (total 16 columns):
 #   Column                                             Non-Null Count   Dtype  
---  ------                                             --------------   -----  
 0   VIN (1-10)                                         280833 non-null  object 
 1   County                                             280821 non-null  object 
 2   City                                               280821 non-null  object 
 3   State                                              280833 non-null  object 
 4   Postal Code                                        280821 non-null  float64
 5   Model Year                                         280833 non-null  int64  
 6   Make                                               280833 non-null  object 
 7   Model                                              280833 non-null  object 
 8   Electric Vehicle Type                              280833 non-null  object

## ETL & Data Cleaning
To make this data actionable, we need to:
1. **Parse Spatial Data:** Extract Latitude and Longitude from the `Vehicle Location` column.
2. **Binary Transformation:** Simplify `Clean Alternative Fuel Vehicle (CAFV) Eligibility` into a binary flag.
3. **Temporal Processing:** Ensure the Model Year is treated correctly for growth analysis.

In [2]:
# 1. Clean Vehicle Location (POINT (Long Lat))
def extract_coords(location):
    if pd.isna(location):
        return None, None
    # Remove 'POINT (' and ')' then split
    clean_loc = location.replace('POINT (', '').replace(')', '')
    long, lat = clean_loc.split(' ')
    return float(lat), float(long)

df['Latitude'], df['Longitude'] = zip(*df['Vehicle Location'].apply(extract_coords))

# 2. Binary CAFV Eligibility Flag
# 'Clean Alternative Fuel Vehicle Eligible' -> 1, others -> 0
df['CAFV_Eligible'] = df['Clean Alternative Fuel Vehicle (CAFV) Eligibility'].apply(
    lambda x: 1 if 'Clean Alternative Fuel Vehicle Eligible' in str(x) else 0
)

# 3. Filter for Washington State (focusing on the utility's regional scope)
df_wa = df[df['State'] == 'WA'].copy()

# Using 'Postal Code' as it is the actual column name in the dataset
display(df_wa[['Postal Code', 'Latitude', 'Longitude', 'CAFV_Eligible']].head())

,Postal Code,Latitude,Longitude,CAFV_Eligible
0,98034.0,47.72201,-122.22901,1
1,98110.0,47.62759,-122.52100,1
2,98144.0,47.57874,-122.30866,1
3,98597.0,46.94239,-122.60735,0
4,98030.0,47.37483,-122.19975,1


## Exporting Cleaned Data for Tableau
To build the dashboard in Tableau, we will export the processed `df_wa` DataFrame. This file includes the cleaned Latitude, Longitude, and CAFV eligibility metrics.

In [3]:
import os

# Export cleaned data to CSV for Tableau
export_dir = '../datasets/cleaned_dataset'
os.makedirs(export_dir, exist_ok=True)
export_path = f'{export_dir}/Cleaned_EV_Population_Data.csv'
df_wa.to_csv(export_path, index=False)

print(f"Success! Cleaned data exported to: {export_path}")

Success! Cleaned data exported to: ../datasets/cleaned_dataset/Cleaned_EV_Population_Data.csv
